In [1]:
import os
import dotenv

dotenv.load_dotenv()

if not os.getenv("GITHUB_TOKEN"):
    raise ValueError("GITHUB_TOKEN is not set")

os.environ["OPENAI_API_KEY"] = os.getenv("GITHUB_TOKEN")
os.environ["OPENAI_BASE_URL"] = "https://models.inference.ai.azure.com/"

In [2]:
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import Settings
import os

llm = OpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
)

embed_model = OpenAIEmbedding(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
)

Settings.embed_model = embed_model

In [3]:
from llama_index.core import StorageContext, load_index_from_storage, VectorStoreIndex, SimpleDirectoryReader

try:
    storage_context = StorageContext.from_defaults(
        persist_dir="../local_index"
    )

    index = load_index_from_storage(storage_context)
except:

#Note: we have to reduce the batch size to stay within the token limits of the free se
    documents = SimpleDirectoryReader("data").load_data()
    index = VectorStoreIndex.from_documents (documents, insert_batch_size=150)
    index.storage_context.persist(persist_dir="../local_index")

In [7]:
# solution
query_engine = index.as_query_engine(
  llm=llm
)
query_engine.query("rag")

Response(response='Retrieval-Augmented Generation (RAG) is an AI framework designed to enhance the accuracy of Large Language Models (LLMs). It achieves this by retrieving relevant and current information from reliable external sources, such as databases or documents, prior to generating a response. This approach helps minimize inaccuracies, often referred to as hallucinations, and allows for the inclusion of citations in the answers provided. Additionally, RAG serves as a cost-effective solution compared to the retraining of models.', source_nodes=[NodeWithScore(node=TextNode(id_='0e459fc5-d1eb-41e6-98f4-1c4f846f7117', embedding=None, metadata={'file_path': '/workspaces/introduction_to_Retrival_Ugmented_generation/data/data.txt', 'file_name': 'data.txt', 'file_type': 'text/plain', 'file_size': 360, 'creation_date': '2026-02-01', 'last_modified_date': '2026-02-01'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accesse

In [9]:
query_engine.query("what are trusted sources")

Response(response='Trusted sources refer to reliable and authoritative databases or documents from which relevant and up-to-date information can be fetched. These sources are used to enhance the accuracy of responses generated by AI frameworks.', source_nodes=[NodeWithScore(node=TextNode(id_='0e459fc5-d1eb-41e6-98f4-1c4f846f7117', embedding=None, metadata={'file_path': '/workspaces/introduction_to_Retrival_Ugmented_generation/data/data.txt', 'file_name': 'data.txt', 'file_type': 'text/plain', 'file_size': 360, 'creation_date': '2026-02-01', 'last_modified_date': '2026-02-01'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='87248e73-dcaa-4ed8-954d-88115fc7eb01', node_type='4', metadata={'file_path': '/works